# Python template
- 基本的なコード、よく使うものなどの備忘録

In [ ]:
# import
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

# from causalimpact import CausalImpact
# from prophet import Prophet
# import pymc as pm
# import arviz as az

import sklearn

# create datetime
print("Created(JST)", pd.Timestamp.now(tz="Asia/Tokyo").strftime("%Y-%m-%d %H:%M:%S"))

# version
print("pandas", pd.__version__)
print("numpy", np.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("seaborn", sns.__version__)
# import causalimpact; print("causalimpact", causalimpact.__version__)
# import prophet; print("prophet", prophet.__version__)
# print("pymc", pm.__version__)
# print("arviz", az.__version__)

print("scikit-learn", sklearn.__version__)

Created(JST) 2025-10-30 08:59:41
pandas 2.2.0
numpy 1.26.4
matplotlib 3.10.5
seaborn 0.13.2


# read data

In [ ]:
# # read data
# # xlsx file
# df1 = pd.read_excel("data.xlsx", 
#                     sheet_name="Sheet1",
#                     skiprows=4,  # skip first 4 rows エクセルでの行番号を指定する　header=Noneの場合はheader行も含めてカウントする
#                     usecols="A:C",  # use columns A to C  エクセルでの列名で指定する　必要列が飛び飛びの場合はリストで指定する usecols=['A','C','D'] 飛び飛びでもこれでよい usecols='A:C','F:I'
#                     header=None,  # no header
#                     names=["date", "value1", "value2"],  # assign column names  カラム名をネーミングする
#                     nrows=100,  # read first 100 rows  最初のn行だけ読み込む （4行スキップの上で）最初の100行を読み込む、必要データの下部にもなにかデータがあると全部読み込んでしまうので切り捨てている
#                     )


# # csv, tsv, txt file
# df1 = pd.read_csv("data.csv", header=0, sep="\t")

In [ ]:
# # 不要レコードがあった場合に削除して上書き
# df1 = df1[df1["date"]<=20231231]
# # df1 = df1.dropna()
# # df1 = df1.query("value1 > 0")

In [ ]:
# サンプルのdataframeを生成する

import string

# 乱数の固定
np.random.seed(42)

# 基本設定
n_rows = 200
n_customers = 50 
weekdays = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
menu = ['Coffee', 'Tea', 'Cake', 'Sandwich']
sections = ['Indoor', 'Terrace']
genders = ['Male', 'Female', 'Other']

# 1. 【マスター作成】50人分のユニークな属性を決めてしまう
chars = string.ascii_lowercase + string.digits
customer_pool = [''.join(np.random.choice(list(chars), 8)) for _ in range(n_customers)]

customer_master = pd.DataFrame({
    'customer_id': customer_pool,
    'customer_gender': np.random.choice(genders, n_customers),
    'customer_age': np.random.normal(35, 12, n_customers).clip(18, 70).astype(int)
})

# 2. 【履歴作成】200行分の「誰がいつ何をしたか」の枠組みを作る
df1 = pd.DataFrame({
    'customer_id': np.random.choice(customer_pool, n_rows, replace=True),
    'date': np.random.choice(pd.date_range('2025-07-01', periods=7), n_rows),
    'menu': np.random.choice(menu, n_rows),
    'section': np.random.choice(sections, n_rows),
    'price': np.random.randint(400, 1200, n_rows),
    'satisfaction': np.random.uniform(1, 5, n_rows)
})

# 3. 【結合】IDをキーにしてマスターの属性をドッキングさせる
df1 = pd.merge(df1, customer_master, on='customer_id', how='left')

# 仕上げ：順序や型を整える
df1['weekday'] = df1['date'].dt.strftime('%a') # 日付から自動で曜日を生成
df1['weekday'] = pd.Categorical(df1['weekday'], categories=weekdays, ordered=True)
df1['menu'] = pd.Categorical(df1['menu'], categories=menu, ordered=True)

In [ ]:
# read data check
print(df1.info())
print(df1.head())

# print(df1.shape)
# print(df1.describe())

In [ ]:
# ざっと見たい時にカラムが多いと見づらいので一時的に表示カラム数を増やす
# pd.get_option('display.max_columns')  # default is 20

# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_columns', 20)  # デフォルトに戻す


# object型のカラムを確認したいが多すぎて全部表示しきれない時は、これで全部表示してくれる
df1.dtypes


# 数値型以外のだけを出力する場合
df1.select_dtypes(exclude=["number"]).info()

# object型のだけでよければ
df1.select_dtypes(include=["object"]).info()
# int型のだけでよければ
df1.select_dtypes(include=["int64"]).info()
# float型のだけでよければ
df1.select_dtypes(include=["float64"]).info()

# データセットの整形

In [ ]:
# # objectになった日付データをdatetime型に変換する例
# df1["date"] = pd.to_datetime(df1["date"])

# # int64型になった日付データをdatetime型に変換する例　一度文字列にしてから日付に変換
# df1["date"] = pd.to_datetime(df1["date"].astype(str))

# # object型の数値データをfloat型に変換する例
# df1["value1"] = df1["value1"].astype(float)

# df1["value1"] = df1["value1"].astype(Int64)  # 小文字でint64と書くとエラーになる

In [ ]:
# 数値型の列（int系とfloat系）を抽出して、すべて float64 に統一
float_cols = df1.select_dtypes(include=['int64', 'int32', 'float64', 'float32']).columns
df1[float_cols] = df1[float_cols].astype(float)

In [ ]:
# # 大きな数値を桁上げしたい
# df1["value1_million"] = df1["value1"] / (10**6)  # 百万で割る
# df1["value1_thousand"] = df1["value1"] / (10**3)  # 千で割る

In [ ]:
# # 0除算でinfやNaNになってる箇所を置換する
# df1["value2"] = df1["value2"].replace(np.inf, 0).fillna(0)

## read data check
- データセットの中身がよくわからないとき

In [ ]:
# min, max確認
print(df1["value1"].min())
print(df1["value1"].max())
df1["value1"].max() - df1["value1"].min() + pd.Timedelta(days=1)  # 日付データの場合の範囲計算例

# meadn, median確認
print(df1["value1"].mean(numeric_only=True))
print(df1["value1"].median(numeric_only=True))

# count
print(df1["value1"].count())

# distinctで一覧を眺める、階層の種類を確認する
print(df1["value1"].unique())

# 上のを並べ替えした状態で眺めたかったら、numpyのsort機能を使う
print(np.sort(df1["value1"].unique()))

# ユニークカウント
print(df1["value1"].nunique())

# どんなデータが多いかをcountして眺める
print(df1["value1"].value_counts())  # デフォルトは降順ソート

# 各変数の欠測値数を確認する
print(df1.isnull().sum())  # 各カラムごとの欠測値数
print(df1["value1"].isnull().sum())  # 特定カラムの欠測値数
# ただしこれだと空白や?が入ってるものは見つけられない、ので次のようにする
print(df1.isin(["", "?", "NA"]).sum())  # 各カラムごとの特定値数

# あるいはsortして眺める
df1["value1"].sort_values(ascending=False).head()  # 降順
df1["value1"].sort_values().head()  # 昇順（デフォルト）

# 特異データチェック
print(df1.query("value1 < 0"))  # value1が0未満のデータを抽出して眺める

# percentile
np.percentile(df1["value1"], 90)
# NaNがたくさんあるデータだと、NaNはめちゃ大きい値として扱われるため90ptがNaNで返る時ある、そんな時は
np.nanpercentile(df1["value1"], 90)

# カラム内の値を置き換えたい場合
df1["value1"] = df1["value1"].replace(["", "?"], np.nan)  # 空白と?をNaNに置換

# 複数置換例
# df1["value1"] = df1["value1"].replace({9999: np.nan, 8888: 0})  # 複数置換辞書指定
# df1["value1"] = df1["value1"].replace(["", "?"], [np.nan, 0])  # 複数置換リスト指定

# dataframeの特定カラムをlistにする
value1_list = df1["value1"].tolist()

## カラムの新規追加

In [ ]:
# 抽出し忘れてたカラムを新規追加
df1["new_column"] = "default_value"  # すべての行に同じ値を入れる（applyのように要素ごとに処理する必要がない場合）
df1["new_column"] = df1["value1"] * 2  # 既存カラムを使って新規カラムを作成する場合

## apply, assign
- SeriesやDataFrameの各要素に対して関数を順番に適用する
- applyはlambdaなどと併用してよく使う

In [ ]:
# prophetでよく使うのは、繁忙期定義用の関数用意して、dsカラムで判定して、on_seasonカラムに繁忙期ornotを1/0で入れる
def on_season(ds):
    date = pd.to_datetime(ds)
    if (date.month == 12) or (date.month == 1):
        return 1
    else:
        return 0

df1["on_season"] = df1["ds"].apply(on_season)
# applyは要素ごとに処理をしていく


# あるカラムでis_hogeの識別をする
df1["is_hoge"] = df1["category"].apply(lambda x: 1 if x == "hoge" else 0)



# assignは新規カラムを追加する時などに
# df1 = df1.assign(value1_squared = df1["value1"] ** 2)  # value1の二乗を新規カラムとして追加
# df1["new_column"] = df1["value1"] * 2  でもいいわけだが、assignは連鎖的に使えるので便利な場合がある、かも、しれない。。

## カラム名の変更

In [ ]:
# renameカラム名の変更
df1 = df1.rename(columns={"value1": "new_value1", "value2": "new_value2"})
# df1.columns = ["col1", "col2", "col3"] でもいいが、renameの方が特定カラムだけ変更できるので便利

# 辞書内包方式でまとめる
# ratioがついているカラムだけ置換する。他はそのまま残る
df1 = df1.rename(columns={col: col.replace("ratio", "") for col in df1.columns if "ratio" in col})
# 全てのカラムに対して、replaceを実行する場合は以下のようにする。ratioがないカラムはそのままだから安心
df1 = df1.rename(columns=lambda x: x.replace("ratio", ""), inplace=True)
# dataframe全体で列名を一括置換する。regex=Falseを付けると単純な文字列置換になる
df1.columns = df1.columns.str.replace("ratio", "", regex=False)

## dropna

In [ ]:
# pandasのdropnaは欠損値NaNがある行を落とす
df1 = df1.dropna()  # いずれかの列にNaNがあれば、行ごと削除（デフォルトは行方向）
# df1 = df1.dropna(axis=1)  # いずれかの行にNaNがあれば、列ごと削除
# df1 = df1.dropna(thresh=3)  # NaNでない値が3つ以上ある行を残す
# df1 = df1.dropna(subset=["value1", "value2"])  # 特定カラムだけを見て、NaN判定する
# subsetは欠損値を含む行を除外するための引数、これを指定することで特定カラムに欠損値がある行を除外してい集計できたりする、たぶん

## is_after

In [ ]:
# pre/post区分やOLS用に0/1フラグ（is_afterカラム）を付与する

# 日付指定で振り分ける
base_date = pd.to_datetime("2023-10-01")
df1["is_after"] = df1["date"].apply(lambda x: 1 if x >= base_date else 0)

# is_afterをdfの末尾7daysにしたければ
base_date = df1["date"].max() - pd.Timedelta(days=6)
df1["is_after"] = df1["date"].apply(lambda x: 1 if x >= base_date else 0)

# is_afterの基準日が複数ある場合
base_date_a = pd.to_datetime("2023-10-01")
base_date_b = pd.to_datetime("2023-11-01")
def is_after_multiple(ds):
    date = pd.to_datetime(ds)
    if date >= base_date_b:
        return 2
    elif date >= base_date_a:
        return 1
    else:
        return 0

## 土日祝フラグ

In [ ]:
# これだと土日フラグなので、
df1["is_holiday"] = df1["date"].apply(lambda x: 1 if x.weekday() >=5 else 0)  # 土日祝日フラグ、土日なら1、平日なら0

# 祝日リストを用意しておいて、
holiday_list = pd.to_datetime(["2023-01-01", "2023-01-02",
                               "2023-02-11", "2023-04-29",
                               "2023-05-03", "2023-05-04"])
# 祝日を1に上書きする
df1.loc[df1["date"].isin(holiday_list), "is_holiday"] = 1

In [ ]:
# 土日祝日フラグを追加する

# 1. 祝日データの作成
import jpholiday

# df['ds'] から期間の開始日と終了日を取得
start = df['ds'].min()
end = df['ds'].max()
# または手打ち
# start = pd.to_datetime("2025-07-01")
# end = pd.to_datetime("2025-07-31")

# 年をまたぐ可能性考えて、ユニークな年を取得
years = df['ds'].dt.year.unique()

holidays_list = []
for y in years:
    for date, name in jpholiday.year_holidays(int(y)):
        # データの期間内に入っている祝日だけ追加
        if start.date() <= date <= end.date():
            holidays_list.append({'ds': pd.to_datetime(date), 'holiday': name})

holidays_df = pd.DataFrame(holidays_list)
holidays_df


# 2. 土日フラグを立てておいて（5:土曜日, 6:日曜日）
df["is_holiday"] = df["ds"].dt.weekday.apply(lambda x: 1 if x >= 5 else 0)

# 3. holidays_df に含まれる日付（平日祝日）を 1 に更新する
# dt.normalize() を使うと時刻が 00:00:00 に揃うので比較が確実
holiday_dates = holidays_df['ds'].dt.normalize()
df.loc[df['ds'].dt.normalize().isin(holiday_dates), "is_holiday"] = 1

## 日付の操作

In [ ]:
# 特定の日付を格納しておいて、後で参照したい
# 日付データとして認識されてないと苦労が多いので、pd.to_datetimeして格納する
base_date = pd.to_datetime("2023-10-01")

# 日付の引き算
# こうしておいて（choose banchmark period）
t_minus_1 = pd.to_datetime(f"{base_date}")

# benchmart週を（base_date, -1週目=介入直前週）を基点として、何週目にあたるか
week_list['point'] = ((week_list['weekly'] - t_minus_1 - datetime.timedelta(days=7)).dt.days / 7 ).astype(int)


test_date = pd.to_datetime("2024-12-25")
# これの1日前が欲しかったら、
previous_date = test_date - pd.Timedelta(days=1)

# データ型を確かめたいときは
type(previous_date)

In [ ]:
# 日付を格納しておいて
base_date = pd.to_datetime("2024-12-03")
oct_first_date = pd.to_datetime("2024-10-01")
oct_last_date = pd.to_datetime("2024-10-31")

# 格納した日付の前日をtimedeltaで計算して、そこまでの平均線を引く
mean_oct = df1.query('@oct_first_date<=date<=@oct_last_date')['revenue'].mean()
plt.hlines(y=mean_oct, xmin=oct_first_date+pd.Timedelta(days=1), xmax=oct_last_date-pd.Timedelta(days=1), colors='gray', linestyles='dotted', linewidth=1.5, label="mean_October")

In [ ]:
# prophetでよくやるのは、
# 後でグラフ表示などに使うために、test期間の初日の日付のindexを格納しておいて
test_date = pd.to_datetime("2024-12-10")
test_date_index = df1.query('ds==@test_date').index[0]
print(test_date_index)

# 後でモデルを評価するために、観測データをtrain,testに分割しておく
df_train = df[: test_date_index]
df_test = df[test_date_index :]
print(df_train.head())
print(df_test.head())

# とやっておいてグラフ表記の時に使ったり
# test期間の点推定での予測値と区間推定での予測値（80%信用区間）を取り出す
df_predict = pd.DataFrame(predict)
df_predict[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].query('ds >= @test_date')

# 実測値と予測値と80%CIを結合してちょい加工する
df_yhat = pd.merge(df_test[['ds', 'y']], df_predict[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].query('ds >= @test_date'), on='ds')
df_yhat.set_index('ds', inplace=True)

NameError: name 'df1' is not defined

In [ ]:
# 時点カラムを整数値で表現したカラムを2種用意する

# 1つめ　データセットの最古時点を1始まりとした（絶対的な）数値カラムを新設
df['absolute_ds'] = df['date'].rank(method='dense').astype(int)

# 2つめ　介入開始時点の1時点前を基準0とした（相対的な）数値カラムを新設
# 辞典が全て連続しているデータセットの場合は、基準時点からの相対的な自転番号を取得、で済むが（日次での例）
baseline_point = pd.to_datetime("2023-10-01")
df['relative_ds'] = (df['date'] - baseline_point).dt.days

# 連続してないデータセットの場合は、absolute_dsを使って相対的な自転番号を取得する
baseline_point = pd.to_datetime("2023-10-01")
df['relative_ds'] = df['absolute_ds'] - df['absolute_ds'].loc[df['date'] == baseline_point].values[0] + 1
# df['relative_ds'] = df['absolute_ds'] - 16 # before期間が16時点あって、基準時点が17時点目だった場合（ほんとか？）

# 意図した通りにナンバリングできているかチェック
pd.pivot_table(df, index='date', values=['absolute_ds', 'relative_ds'] ,aggfunc='count')

In [ ]:
# fiscal year単位で集計するために用意
def fiscal_year(ds):
    date = pd.to_datetime(ds)
    if date.month >= 4:
        return date.year
    else:
        return date.year - 1
df1["fiscal_year"] = df1["date"].apply(fiscal_year)

# 4月開始
y = df1["date"].dt.year
m = df1["date"].dt.month
df1["fiscal_year"] = np.where(m >=4, y, y-1)

# 頭にFYをつけたラベルを作る
df1["fiscal_year_label"] = "FY" + df1["fiscal_year"].astype(str).zfill(4)  # zfill(4)で4桁に0埋め FY2023
df1["fiscal_year_label"] = "FY" + df1["fiscal_year"].astype(str).str[-2:]  # FY23

## カテゴリ変数の並び順を指定しておく

In [ ]:
# 曜日を日付カラムから取得する、でもある
# 曜日カラムの作成
df1["weekday"] = df1["date"].dt.day_name()
# これだと文字列で曜日が入る。Monday, Tuesday, Wednesday, Thursday, Friday, Saturday, Sunday

# カテゴリ変数の並び順を指定しておく
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df1["weekday"] = pd.Categorical(df1["weekday"], categories=weekday_order, ordered=True)

# 数値表現の曜日カラムを作成する場合
df1["weekday_num"] = df1["date"].dt.weekday  # 0=Monday, 6=Sunday
df1["weekday_num"] = df1["date"].dt.weekday + 1  # 1足して1=Monday, 7=Sundayにする

In [ ]:
# month_shortカラムをカテゴリ変数として並び順指定しておく
month_short_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
df1["month_short"] = df1["date"].dt.strftime("%b")  # Jan, Feb, Mar...
df1["month_short"] = pd.Categorical(df1["month_short"], categories=month_short_order, ordered=True)

# FYの順序を指定しておいて
fy_order = ["FY22", "FY23", "FY24", "FY25"]
df1["fiscal_year_label"] = pd.Categorical(df1["fiscal_year_label"], categories=fy_order, ordered=True)

In [ ]:
# pd.Categorical(...) を使う場合（上のやり方）
# データそのものを変換するというアクション（動詞）に近いイメージ

# CategoricalDtype を使う場合（下のやり方）
# 型（ルール）を定義するという設計図（名詞）を作るイメージ

# 今ある値を重複なく取ってきて、アルファベット順にソートする
store_order = sorted(df1['store_id'].unique())

# 1. 理想の並び順をリストで作る
store_order = [f'Store_{c}' for c in 'ABCDEFGHIJ'] 
# もし手動なら ['Store_A', 'Store_B', ..., 'Store_J'] って書いてもOK！

# 2. 順番（ordered=True）を指定してカテゴリ型に変換
from pandas.api.types import CategoricalDtype
store_type = CategoricalDtype(categories=store_order, ordered=True)
df1['store_id'] = df1['store_id'].astype(store_type)

# なぜ CategoricalDtype の方が「丁寧」と言われるのか？
# 機械学習のパイプラインを作るときは、CategoricalDtype を使うほうがトラブルが少なかったりする。
# 例えば、テストデータに「Store J」のデータがたまたま1件も入っていなかった場合：
# pd.Categorical だと、その時のデータにある店舗だけでカテゴリを作ろうとしちゃうことがある。
# CategoricalDtype で「Jまであるよ」という設計図をastypeで流し込めば、データがなくても「Store J（データ数0）」として枠組みを維持してくれる

## 複数カラムを文字列として連結して新しいカラムを

In [ ]:
# ナンバリングのカラムと名前のカラムをくっつけて、結合
df1["combined_column"] = df1["number_column"].astype(str) + "_" + df1["name_column"]

# 数値の前にゼロをつけて2桁にしてから結合する
df1["combined_column"] = df1["number_column"].astype(str).str.zfill(2) + "_" + df1["name_column"]

## 性年代区分を追加

In [ ]:
# 性年代区分を追加
# 関数を用意してapplyで各行に適用する
def age_gender_category(row):
    age = row['age']    gender = row['gender']
    if gender == 'Male':
        if age < 20:
            return 'Male_U20'
        elif age < 30:
            return 'Male_20s'
        elif age < 40:
            return 'Male_30s'
        elif age < 50:
            return 'Male_40s'
        elif age < 60:
            return 'Male_50s'
        else:
            return 'Male_60plus'
    elif gender == 'Female':
        if age < 20:
            return 'Female_U20'
        elif age < 30:
            return 'Female_20s'
        elif age < 40:
            return 'Female_30s'
        elif age < 50:
            return 'Female_40s'
        elif age < 60:
            return 'Female_50s'
        else:
            return 'Female_60plus'
        
df1['age_gender_category'] = df1.apply(age_gender_category, axis=1)


# andで記述する場合
def age_gender_category(row):
    age = row['age']    gender = row['gender']
    if (gender == 'Male') & (age < 20):
        return 'Male_U20'
    elif (gender == 'Male') & (age < 30):
        return 'Male_20s'
    elif (gender == 'Male') & (age < 40):
        return 'Male_30s'
    elif (gender == 'Male') & (age < 50):
        return 'Male_40s'
    elif (gender == 'Male') & (age < 60):
        return 'Male_50s'
    elif (gender == 'Male'):
        return 'Male_60plus'
    elif (gender == 'Female') & (age < 20):
        return 'Female_U20'
    elif (gender == 'Female') & (age < 30):
        return 'Female_20s'
    elif (gender == 'Female') & (age < 40):
        return 'Female_30s'
    elif (gender == 'Female') & (age < 50):
        return 'Female_40s'
    elif (gender == 'Female') & (age < 60):
        return 'Female_50s'
    else:
        return 'Female_60plus'

df1['age_gender_category'] = df1.apply(age_gender_category, axis=1)

# 順序を指定しておいて
age_gender_order = ['Male_U20', 'Male_20s', 'Male_30s', 'Male_40s', 'Male_50s', 'Male_60plus',
                    'Female_U20', 'Female_20s', 'Female_30s', 'Female_40s', 'Female_50s', 'Female_60plus']
df1['age_gender_category'] = pd.Categorical(df1['age_gender_category'], categories=age_gender_order, ordered=True)

## 対数変換

In [ ]:
# 対数変換する、そして指数関数で元のscaleに戻す
df1['log_value1'] = np.log1p(df1['value1'])  # log1pはlog(1+x)なので、value1が0でもエラーにならない
df1['exp_value1'] = np.expm1(df1['log_value1'])  # expm1はexp(x)-1なので、log1pで変換したものを元に戻せる

# 自然対数で変換してる（常用対数でやりたい場合はnp.log10）
df1['revenue_thousand_log'] = np.log(df1['revenue_thousand'] + 1)  # revenue_thousandが0でもエラーにならないように1足してからlog変換

# 元のscaleに戻すときは
# 自然対数の時は、np.exp(x)で戻す（
df1['revenue_thousand_exp'] = np.exp(df1['revenue_thousand_log']) - 1  # 1引いて元のscaleに戻す

# 常用対数の時は10**xで戻す、あるいはnp.power(10, x)で戻す
original_scale = 10 ** df1['revenue_thousand_log'] - 1  # 1引いて元のscaleに戻す

# グラフなどで対数変換した値であることを表現するなら
# 例 ln(Revenue + 1), log10(Revenue + 1) などと表記する
# 例 Trend of Revenue (Log Scale)
# 例 注釈：データの分布を平滑化するために、Revenueに1を足して自然対数変換（ln）しています。



# 対数変換した値での回帰分析の結果の解釈の仕方（ここではDIDの場合）
# 例えば、対数変換（するときに+1）した売上(log_revenue)を目的変数にして、介入効果を推定した場合

# 交互作用項が介入効果を表すとして、0→1に1単位変化させたとき（処置群のbefore→after）に
# 結果変数であるlog_revenueがどれだけ変化するかを示す係数が得られる
# 例えば、介入効果の係数が0.05だったとします
# これは、log_revenueが0.05増加したことを意味します
# ここで、log_revenueはlog(revenue + 1)で定義されているので、
# revenue + 1 = exp(log_revenue) という関係から、
# revenue + 1 = exp(original_log_revenue + 0.05) = exp(original_log_revenue) * exp(0.05)
# となります
# つまり、revenue + 1がexp(0.05)倍になることを意味します
# したがって、revenue自体の変化率は
# (revenue_new - revenue_old) / revenue_old = (exp(0.05) - 1)
# となり、約5.13%の増加を意味します
# 具体的には、exp(0.05) - 1 ≈ 0.0513、つまり約5.13%の増加です

# 介入効果の係数が負の値だった場合も同様に解釈できます
# 例えば、係数が-0.03だった場合、exp(-0.03) - 1 ≈ -0.0296、つまり約2.96%の減少を意味します

## 一定のpercentileを丸める

In [ ]:
# 一定のpercentileを丸める
# 例えば、value1カラムを99percentileに丸める場合
percentile_99 = np.nanpercentile(df1['value1'], 99)
df1['value1_capped'] = np.where(df1['value1'] > percentile_99, percentile_99, df1['value1'])

# 複数カラムを一括で丸める場合

# 任意のpercentileで丸める関数
def round_to_pt(groups, columns, percentile):
    for column in columns:
        pt_num = np.nanpercentile(groups[column].dropna(), percentile)  # NaN除外してpercentile計算
        groups[column] = groups[column].apply(lambda x: min(x, pt_num) if not np.isnan(x) else x)
    return groups

# 丸めたいカラムのリスト
columns_to_round = ['value1', 'value2', 'value3']

# 適用
df1[columns_to_round] = round_to_pt(df1[columns_to_round], columns_to_round, 90)


# ごく稀に、上記を適用しても、丸めた値よりも大きな値が出現する場合がある　interolationに起因するらしい
# np.percentile関数のinterpolationパラメーターのデフォルト設定が、Numpy1.22以降では"linear"から"nearest"に変更されたため
# linear 位置が整数でない場合、隣接する2つの値の線形補間を使用してパーセンタイル値を計算します
# nearest 位置が整数でない場合、最も近いデータポイントを使用してパーセンタイル値を計算します
# lower 位置が整数でない場合、下側（直前、小さい方）のデータポイントを使用してパーセンタイル値を計算します
# higher 位置が整数でない場合、上側（直後、大きい方）のデータポイントを使用してパーセンタイル値を計算します
# midpoint 位置が整数でない場合、隣接する2つの値の平均を使用してパーセンタイル値を計算します

In [ ]:
# percentile指定じゃなくて、具体的な値で丸めたい場合は、clipを使うと便利
# 例えば、value1カラムを下限0に、上限100で丸める場合
df1['value1_capped'] = df1['value1'].clip(lower=0, upper=100)  # 下限0 上限100
# NaNはそのまま残る（置換の対象外）

# clipは切る、切り取る、端を削る、という意味

## 一定のpercentileを除外する

In [ ]:
# 一定のpercentileを除外する
# 基本は丸める方針（サンプルサイズ維持を企図している）なんだが、
# ARPUのようなゼロ箇条分布なのを、ゼロ過剰モデル、確率分布Dはガンマ分布でベイズ推定する場合などは、
# 丸めるとかえってガンマ分布が近似しにくくなって,mcmcが収束しない、終わらない場合がある
# なので、どうにも裾の長さが影響してそうなら、一定のpercentileを除外するとき用

def remove_outliers(df, group_column, target_column, percentile=95, interpolation='nearest'):
    # 対象カラムが全てNaNの場合はそのまま返す
    def filter_outliers(group):
        if group[target_column].isnull().all:
            return group
        # NaNを除外してpercentile計算
        values = group[target_column].dropna().values
        # np.percentileのinterpolationパラメーターを指定してpercentile計算
        threshold = np.percentile(values, percentile, interpolation=interpolation)
        return group[group[target_column] <= threshold]
    return df.groupby(group_column).apply(filter_outliers).reset_index(drop=True)

df1 = remove_outliers(df1, group_column='category', target_column='value1', percentile=95, interpolation='nearest')

# dfの行や列を指定して参照・抽出
- loc = label location 列「名」や行「名」で位置を指定して、複数セルを取り出す
- iloc = integer location  列「番号」や行「番号」で位置を指定して、複数セルを取り出す
- 基本構文は、df.loc[行, 列] df.iloc[行, 列]
  - ,カンマは行と列を、分けるという意味
  - :コロン単独だと「すべてを選択する（すべての行またはすべての列を指定する）」という意味、これで使うことが多いかな
    - start: startから最後の行or列までを指定
    - :end 最初の行or列からendの手前までを指定（endは含まれない）
    - start:end startからendの範囲を指定（endは含まれない）
    - start:end:step startからendの範囲をstepごとに指定（使うかな？）
- at 行「名」と列「名」を両方指定して、1セルを取り出す
- iat 行「番号」と列「番号」を両方指定して、1セルを取り出す

In [ ]:
# すべての行の、revenueカラムのみを切り出す
df1.loc[:, ['revenue']].head()
# hoge行のみの、すべてのカラムを切り出す
df1.loc[df1['hoge'], :].head()

# すべての行の、0番目のカラムのみを切り出す
df1.iloc[:, [0]].head()
# Seriesに対してilocを使う場合
series1['value1'].iloc[0]  # そのseriesの0番目の要素を取得する（コロンを使わないと単に一つの要素になる）


df1.at[df1.index[0], 'value1']  # 最初の行のvalue1
df1.at[2, 'value1']  # 3行目のvalue1

# データを取得するだけでなく、更新（新たな値を設定、代入、上書き）もできる
df1.at[df1.index[0], 'value1'] = 100  # 最初の行のvalue1を100に更新
df1.at[2, 'value1'] = None  # 3行目のvalue1をNoneに更新

# query

In [ ]:
# 一定の条件を満たすレコードだけを抽出
df1.query("value1 > 1000 and value2 < 500")
# 条件指定の仕方
# NOT != 
# AND & 
# OR |
# ORはinでもできる（大文字INだとダメ）
# df1.query("category in ['A', 'B', 'C'] and value1 >= 100")
# df1.query("category not in ['A', 'B', 'C'] or value2 < 50")
# df1.query("date >= '2023-01-01' and date <= '2023-12-31'")
# df1.query("@start_date <= date <= @end_date")  # 変数参照する場合は@をつける
# df1.query("value1 == value2")  # 別カラム同士の比較もできる
# df1.query("value1 + value2 > 1000")  # 計算式も使える

# ANDやORの優先づけはカッコで
df1.query("(value1 > 1000 and value2 < 500) or category == 'A'")

# indexで条件指定したい時
df1.query("index >= 10 and index < 20")  # index参照する場合はindexキーワードを使う

# 変数名にスペースやドットが含まれている場合は、ブールインデックスで抽出する
df1[df1['column 1'] > 1 & df1['column.2'] < 6]

# 条件分岐

## if~else (case whenみたいな)

In [ ]:
# case when的な（2区分）
df1['new_column'] = np.where(df1['value1'] > 1000, 'High', 'Low')

# 決定木（分類木）で結果変数をカテゴリ化したい場合
# 基準点を決めておいて
np1 = 0
# 条件分岐の関数を用意してapplyで各行に適用する
def categorize_based_on_np(row):
    np_value = row['np']
    if np_value >= np1:
        return 'stay_or_higher'
    else:
        return 'lower'
df1['np_category'] = df1.apply(categorize_based_on_np, axis=1)


# 3区分以上の場合
# 条件分岐の関数を用意してapplyで各行に適用する
def categorize_based_on_np(row):
    np_value = row['np']
    if np_value > np1:
        return 'Positive'
    elif np_value < np1:
        return 'Negative'
    else:
        return 'Neutral'
df1['np_category'] = df1.apply(categorize_based_on_np, axis=1)


## np.where

In [ ]:
# np.whereは三項演算子（if~ellse）のように捉えるとわかりやすい、かもしれない
# np.where(condition, x, y)
# conditionがTrueならxを返し、Falseならyを返す
# これを行ごとに判定してくれる

# 例えば、value1が1000より大きい場合はvalue1を2倍にし、小さい場合はvalue1を半分にする場合
df1['adjusted_value1'] = np.where(df1['value1'] > 1000, df1['value1'] * 2, df1['value1'] / 2)

# 要約統計量 describe

In [ ]:
df1.describe()

# 四分位数だけでなく90percentileや99percentileも一緒に出したい場合
df1.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99])

# 全カラムの要約統計量は不要だが、つらつら書くと見づらい時は
num_cols = ['revenue', 'value1', 'value2']
df1[num_cols].describe()

# 単一カラムのdecribeを見たい時に、どの順番で書くんだ
df1['value1'].describe()
df1.describe()['value1']
# 結果は同じ。絞り込んでるから前者がちょっと速いかも

# groupby
## groupbyしたときに、groupbyで指定したカラムでsort

In [ ]:
# groupbyしたときにsortしたい
# groupbyで指定したカラムがindexになるので、基本はsort_index

# groupbyが単一（indexが1つ）の場合
df1.groupby('category')['value1'].mean().sort_index(ascending=False)

# groupbyが複数（indexが2つ以上）の場合
df1.groupby(['category', 'subcategory'])['value1'].mean().sort_index(level=['category', 'subcategory'], ascending=False)
# この場合、groupbyの2コとも降順になる

# 1コめは昇順で、2コめは降順にしたいならascending=[True, False]とリストで指定する
df1.groupby(['category', 'subcategory'])['value1'].mean().sort_index(level=['category', 'subcategory'], ascending=[True, False])


# groupbyしたときにクセでreset_index()をつけた状態だと、当然ながらgroupbyで指定したカラムがindexではなくなる
# なので、その場合はsort_valuesで指定する
df1.groupby('category')['value1'].mean().reset_index().sort_values(by='category', ascending=False)
# groupbyが複数（indexが2つ以上）の場合
df1.groupby(['category', 'subcategory'])['value1'].mean().reset_index().sort_values(by=['category', 'subcategory'], ascending=[True, False])

# describeじゃなくてsum()などの時も同様の書き方でよい
df1.groupby(['category', 'subcategory'])['value1'].sum().reset_index().sort_values(by=['category', 'subcategory'], ascending=[True, False])

# aggでgroupby集計したときも同様
df1.groupby('category').agg({'value1': 'mean', 'value2': 'sum'}).reset_index().sort_values(by='category', ascending=False)

## groupbyした時に、集計対象のカラムでsort

In [ ]:
# groupbyした時に、集計対象のカラムでsort
# 上ですでに登場してる通り、sort_valuesを使う
df1.groupby('category')['value1'].describe().sort_values(by='mean', ascending=False)
# 集計対象のカラムとしては1コなんだが、describeなので複数の統計量が出てくる、なのでsort_valuesでmeanを指定する必要ある

## groupbyで集計や新しいdf作成

In [ ]:
# groupbyはいろんなことができるが、集約するカラムを複数指定してる場合は、カラムがMultiIndexになる
# その後が扱いにくいのでフラットにする
# これはpivot,pivot_tableでも同様

df2 = df1_grouped = df1.groupby(['category', 'subcategory'], observed=True).sum().unstack('subcategory').reset_index()
df2.columns = ['_'.join(col).strip() for col in df2.columns.values]
# これで、category_A_value1, category_A_value2, category_B_value1, category_B_value2...のようなフラットなカラム名になる、たぶん

# multiindexの1段目を落とす
df2.columns = df2.columns.droplevel(0, axis=1)
# multiindexの最後のレベルだけ使う
df2.columns = df2.columns.get_level_values(-1)

In [ ]:
# groupbyでただ集計したい場合（つまりクロス集計）

# 同じカラムをcountとnuniqueで集計したい場合
# df1.groupby('category', observed=True).agg([
#     'value1', 'count',
#     'value1', 'nunique'
#     ])
# これだとnuniqueの方しか出力されない
# ので、別名をつける
df1.groupby('category', observed=True).agg(
    value1_count = ('value1', 'count'),
    value1_nunique = ('value1', 'nunique')
    )


# groupbyで複数を指定しておいて、片方をヨコに展開して、クロス集計風（エクセルのピボットみたいな）に表示したい場合
# unstackでヨコに並べたい列を指定する
df1.groupby(['category', 'subcategory'])['value1'].sum().unstack('subcategory')

# 集計関数をそれぞれ変えたい場合はaggを使う
df1.groupby(['category', 'subcategory']).agg(
    value1_sum = ('value1', 'sum'),
    value2_mean = ('value2', 'mean')
    ).unstack('subcategory')


# groupby + unstackを使うことで、集約しつつ、タテ持ちデータをヨコ持ちデータに変換できる
# タテ→ヨコの時に集約不要ならpivotでもOK

In [ ]:
# crosstabでクロス集計
# メリット　行と列にあたるところはdataframeでなくても、Seriesやリストでもよい
# margins=Trueで 行・列の合計行（Total） を簡単に出せる
pd.crosstab(df1['category'], df1['subcategory'], values=df1['value1'], aggfunc='sum', margins=True, margins_name='Total').sort_values(by='Total', ascending=False)

# dfをタテヨコ変換したい
- 基本は
  - pivot タテ→ヨコ
  - melt ヨコ→タテ　なんだが、
- タテ→ヨコの時に集約する/しないも絡んでくる。だからpivot_table, groupbyと混乱しだす
- ヨコ→タテの時に variableはタテよりはヨコにしたいことが多い。するとpivot, pivot_table, groupbyもセットで登場するから混乱しだす

# dfをヨコ持ちに変換する
- たいてのことはタテ持ちが好都合だろう（seabornで描画する時など）
- だが途中で結果変数と共変量（説明変数）をヨコ持ちにしたい、相関係数を計算したいときなどに出番

## pivot

In [ ]:
# 集約不要でタテ→ヨコに変換したい場合はpivotを使う
# indexがタテ（変換後もそのまま残す列）
# columnsがヨコ（変換後に展開する列、ヨコ持ちにする際の区分け条件としたい列）
# valuesが中身（ヨコ持ちにするカラムを指定する
df1.pivot(index='category', columns='subcategory', values='value1')
# valuesが1コの時はリストで渡さないでおけば、df格納してもmultiindexにならない

# valuesが複数の時はリストで渡す
df1.pivot(index='category', columns='subcategory', values=['value1', 'value2'])

# pivotで作成したdfの並び順は、valuesで指定したカラム名の昇順ソートになるっぽい
# 並び順を指定したい場合は
order = ['subcategory_A', 'subcategory_B', 'subcategory_C']
df_pivot = df1.pivot(index='category', columns='subcategory', values='value1')
df_pivot = df_pivot[order]  # columnsを指定して並び替え

## pivot_table

In [ ]:
# 集約しつつタテ→ヨコに変換したい場合はpivot_tableを使う
df1.pivot_table(index='category', columns='subcategory', values='value1', aggfunc='sum')

# valuesのカラムごとに集約関数を変えたい場合はaggfuncに辞書を渡す
df1.pivot_table(index='category', columns='subcategory', values=['value1', 'value2'], aggfunc={
    'value1': 'sum', 
    'value2': 'mean'})

# pivot_tableの注意点
# 存在しない組み合わせのもできあがっちゃう（カテゴリカル変数カラムをindexに指定した場合など）
# NaNが出てくる場合（存在しない組み合わせなど）はfill_valueで置換できる
df1.pivot_table(index='category', columns='subcategory', values='value1', aggfunc='sum', fill_value=0)

# 回避策としてはこういうのもある
df2['subcategory'] = df2['subcategory'].cat.remove_unused_categories()
# 列がcategory型のまま unusedカテゴリを抱えた状態で集計していると、
# デフォルトでは”存在しないはずのカテゴリ（カウント0のカテゴリ）”まで展開されてしまうため、事前にunusedカテゴリを削除しておく必要ある

# groupbyのときも明示的にobserved=Trueにしておかないと、unusedカテゴリがある場合に同様のことが起きるんだっけ
df1.groupby(['category', 'subcategory'], observed=True)['value1'].sum() 

## ヨコ持ちにして横%の構成費を計算

In [ ]:
# ヨコ%にしたい場合
hoge3_ratio = hoge3.div(hoge3.sum(axis=1), axis=0)  # 行方向に合計を計算して、各要素を行合計で割る
# 行方向に%（横の比率＝行合計で割る）
# hoge3.sum(axis=1) は、hoge3の各行の合計を計算する（ヨコ方向の合計を算出している）
# axis=0 は、行方向インデックスを基準にして割る＝行ごとに割る
# 行内の値をその行合計で割る、だからヨコの%

# タテ%にしたい場合
hoge3_ratio = hoge3.div(hoge3.sum(axis=0), axis=1)  # 列方向に合計を計算して、各要素を列合計で割る
# 列方向に%（縦の比率＝列合計で割る）
# hoge3.sum(axis=0) は、hoge3の各列の合計を計算する（タテ方向の合計を算出している）
# axis=1 は、列ラベルを基準にして割る＝列ごとに割る
# 列内の値をその列合計で割る、だからタテの%

# divとは？
# DataFrame.div(other, axis=..)は「割り算」をおこなうメソッド
# df / other と同じ意味なんだが、divを使うとaxisパラメーターで行方向に割るのか、列方向に割るのかを明示できる


## axisの基本

In [ ]:
# Pythonのpandas, numpyでいうaxisは「どの方向に処理するか」を表す
# axis=0 縦方向
# 行をまたいで処理する（行をつぶして、列ごとに計算）
# イメージ　縦に走っている軸に沿って計算する
# axis=1 横方向
# 列をまたいで処理する（列をつぶして、行ごとに計算）
# イメージ　横に走っている軸に沿って計算する

# 上のdivの件は、タテ%の時に、hoge3_ratio = hoge3.div(hoge3.sum(axis=0), axis=1)
# 2回目のaxis=1なのはなぜ？という感じがするが、
# hoge3.sum(axis=0) のindexは元は列（タテ方向）だったんだが今はヨコ方向になってる、だからaxis=1とする

# dfをタテ持ちに変換する
- タテ持ちでデータを読み込んでることが多いが、
- まれにヨコ持ちで読み込んでて、回帰やグラフ描画時にタテ持ちにしたいときがあるんだ

In [ ]:
# melt
df_melted = pd.melt(df1, id_vars=['category', 'date'], value_vars=['value1', 'value2'], var_name='variable', value_name='value')
# id_vars 変換後もそのまま残す列
# value_vars タテ持ちに変換したい列を指定
# var_name value_varsで指定したカラム名を格納する新しい列の名前を指定（未指定時はカラム名 variableになる）
# value_name 変数の値を格納する新しい列の名前を指定（未指定時はカラム名 valueになる）

# 上のだとvalue_varsはタテ持ち状態なので、若干使いづらい場合がある
# なので、variable列を分割して複数列に展開する
df_melted[['variable_prefix', 'variable_suffix']] = df_melted['variable'].str.split('_', expand=True)
# str.split('_', expand=True) で variable列の文字列を'_'で分割し、複数列に展開する

# だがこの書き方はいまいち読みにくくもある
# なので、これの代わりに、groupby + unstack, pivot, pivot_tablem (crosstab)を使う方が楽かもしれない

# copy

In [ ]:
import copy
df_copy = df.copy()

# queryで絞り込んで新しいdfを作成
df = df1.query("value1 > 1000").reset_index(drop=True).copy()

# 必要なカラムだけをコピーして新しいdfを作成する
df_subset = df1[['date', 'category', 'value1']].copy()

# merge,concat,join（結合）
## merge
- sqlでいうjoin

In [ ]:
# merge
df_merged = pd.merge(df1, df2, on='key_column', how='inner')
# カラム名をキーにして結合する、onでキーにするカラムを指定、howで結合方法を指定
# howの指定方法
# 'inner' 内部結合、共通するキーのみ結合
# 'left' 左外部結合、左のdfの全行を保持し、右のdfから一致するキーの行を結合
# 'right' 右外部結合、右のdfの全行を保持し、左のdfから一致するキーの行を結合
# 'outer' 完全外部結合、両方のdfの全行を保持し、一致するキーの行を結合

# 複数カラムをキーにして結合する場合
df_merged = pd.merge(df1, df2, on=['key_column1', 'key_column2'], how='inner')

# 左右のdfでキーのカラム名が異なる場合
df_merged = pd.merge(df1, df2, left_on='left_key_column', right_on='right_key_column', how='inner')

# mergeではindexをキーにすることもできる
df_merged = pd.merge(df1, df2, left_index=True, right_index=True, how='inner')

# mergeでは左はindexをキーにして、右はカラム名をキーにして結合できる
df_merged = pd.merge(df1, df2, left_index=True, right_on='right_key_column', how='inner')

# mergeの欠点：3コ以上の結合を一度にできない
# 3コ以上のdfを結合したい場合は、mergeをネストして使う、ただし読みにくい
df_merged = pd.merge(pd.merge(df1, df2, on='key_column', how='inner'), df3, on='key_column', how='inner')

## concat
- concatはタテにもヨコにも結合できる
- デフォルトはタテなので、sqlでいうUNIONに近い

In [ ]:
# concat
df_concat = pd.concat([df1, df2], axis=0, ignore_index=True)
# concatでタテに結合
# カラム名をキーにしている（mergeでのONのように指定する必要はない）
# df1とdf2のカラム名が同じ場合は、そのまま連結される
# df1とdf2のカラム名が異なる場合は、存在しないカラムにはNaNが入る

# ignore_index=True 連結後にindexを再設定する（デフォルトはFalseで元のindexを保持する、のでindexの重複は発生しうる）


# concatでヨコに結合
df_concat = pd.concat([df1, df2], axis=1)
df_concat = pd.concat([df1, df2], axis='columns')  # axis='columns'でも同じ意味
# indexをキーにして結合される
# なのでmerge感覚でヨコ結合するときは、indexが一致していることを確認する必要がある

## join
- 名前からするとsqlのJOINっぽいが、concatのヨコ結合同様にindexをキーにしている
- なのでmerge寄りではなく、concatのヨコ結合寄り
- concatのヨコ結合に対する利点は、結合方法を指定できる（left,right,inner,outer）（逆にいうとconcatは自由過ぎる）

In [ ]:
# join
df_joined = df1.join(df2, how='inner')
# デフォルトはleft join(how='left')
# indexをキーにして結合される
# howの指定方法はmergeと同じ


# SeriesとDataframe

In [ ]:
# SeriesとDataFrameの違い
# Seriesは1次元配列、DataFrameは2次元配列（行と列）
# Seriesは単一のデータ型を持つ、DataFrameは複数のデータ型を持つことができる

# Seriesはindexを持つ、DataFrameは行と列の両方にindexを持つ
# Seriesは単一のカラムを持つ、DataFrameは複数のカラムを持つ

# SeriesはDataFrameの一部として扱うことができる、DataFrameは複数のSeriesを組み合わせて作成される
# SeriesはDataFrameの特定のカラムを抽出したものとして作成できる
# 例 df['column_name'] はSeriesを返す

# 機械学習のときなどに、dataframeから単一カラムをXに渡したい時に、
# これで良さそうに見えるがSeriesなのでダメ
# X = df['feature_column']  # Seriesとして抽出
# なので、こう書く必要ある
X = df[['feature_column']]  # DataFrameとして抽出、ダブルブラケットで囲む

# print

In [ ]:
# print整形
# カンマ区切り
print("Revenue (yen):{:,.0f}".format(hoge))

# 空白1行を挟む
print('¥n', end='')  # 空白1行を挟む

# 小数点3桁まで表示
print("Value1: {:.3f}".format(hoge))

# formatを複数使う場合
print("Revenue: {0:,.0f} yen, Value1: {1:.3f}".format(hoge_revenue, hoge_value1))

# export

In [ ]:
# export
df1.to_csv('output.csv', index=False, lineterminator='\n')
# 1行おきに空行になる場合があるため、改行コードを指定しておく

df1.to_excel('output.xlsx', index=False)

# 日本語が文字化けしないように文字コードを指定
df1.to_csv('output_utf8bom.csv', index=False, encoding='utf-8-sig', lineterminator='\n')

## 画像ファイルで保存

In [ ]:
# 分類結果などを画像ファイルで保存
hoge,write_png('C://hoge/output_figure.png')

## .ipynbを.pyに
- VS Codeのメニューバーではなく、開いてる.ipynbのメニューバーの3点リーダーのところにエクスポートがある

# 相関行列

In [ ]:
# サクッと相関行列
df,corr(numeric_only=True)
# カラム指定でやる場合
df[['value1', 'value2', 'value3']].corr().round(3)

# 相関行列をヒートマップとしてプロット
cm_corr = df.corr().round(3)
# 対角成分をNaNにする
# np.fill_diagonal(cm_corr.values, np.nan)  # エラーになる場合あるようだ
for i in range(len(cm_corr)):
    cm_corr.iat[i, i] = np.nan

plt.figure(figsize=(10, 8))
sns.heatmap(cm_corr, annot=True, cmap='coolwarm', fmt='.2f', xticklabels=cm_corr.columns, yticklabels=cm_corr.columns)
plt.show()

# 分割

In [ ]:
# 対応あるような状態のをカラム指定で分割
x_A, x_B = df['group_A'], df['group_B']

# 機械学習的にtrain,testに分割
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df1, test_size=0.2, random_state=42)  # 20%をtestデータにする、random_stateで乱数シードを固定
# テストデータを具体的な数値で指定する場合
train_df, test_df = train_test_split(df1, test_size=1000, random_state=42)  # testデータを1000行にする
# トレーニングデータを50%で指定する場合
train_df, test_df = train_test_split(df1, train_size=0.5, random_state=42)  # 50%をtestデータにする


# アンダーサンプリングするだけならsampleでいいっぽい
dfs = df.sample(n=1000, random_state=42)  # ランダムに1000行抽出
dfs = df.sample(frac=0.1, random_state=42)  # ランダムに10%抽出


# アンダーサンプリング
from sklearn.utils import resample
# 少数クラスを抽出
minority_class = df1[df1['target'] == 1]
# 多数クラスを抽出
majority_class = df1[df1['target'] == 0]
# 多数クラスを少数クラスのサイズに合わせてランダムサンプリング
majority_class_downsampled = resample(majority_class,
                                       replace=False,  # 非復元抽出
                                       n_samples=len(minority_class),  # 少数クラスのサイズに合わせる
                                       random_state=42)  # 乱数シードを固定
# ダウンサンプリングした多数クラスと少数クラスを結合
df_balanced = pd.concat([minority_class, majority_class_downsampled])
# インデックスをリセット
df_balanced = df_balanced.reset_index(drop=True)


# 標準化

In [ ]:
# 単回帰用の標準化
sd_X = df['x1'].std()  # x1の標準偏差
mean_X = df['x1'].mean()  # x1の平均
z_X = (df['x1'] - mean_X) / sd_X  # x1を標準化

# 重回帰用の標準化
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_columns = ['x1', 'x2', 'x3']  # 標準化したいカラムリスト
X_scaled = scaler.fit_transform(df[X_columns])
df_scaled = pd.DataFrame(X_scaled, columns=X_columns)

# ブートストラップ信頼区間

In [ ]:
# ブートストラップ信頼区間
import statistics
from random import choices

# data = [1, 2, 3, 4, 5]
list_data = df1['value1'].tolist()  # NaNを除外してリスト化

# mean
print("Mean: {:,.3f}".format(statistics.mean(list_data)))

# 90%CI
mean100 = sorted([statistics.mean(choices(list_data, k=len(list_data))) for _ in range(100)])
ci_lower_90 = mean100[int(0.05 * len(mean100))]
ci_upper_90 = mean100[int(0.95 * len(mean100))]
print("90% CI: [{:,.3f}, {:,.3f}]".format(ci_lower_90, ci_upper_90))
print("¥n", end='')  # 空白1行を挟む

# 95%CI
# 1000個のリサンプリングにする
mean1000 = sorted([statistics.mean(choices(list_data, k=len(list_data))) for _ in range(1000)])
ci_lower_95 = mean1000[int(0.025 * len(mean1000))]
ci_upper_95 = mean1000[int(0.975 * len(mean1000))]
print("95% CI: [{:,.3f}, {:,.3f}]".format(ci_lower_95, ci_upper_95))
# print("¥n", end='')  #

# 分類問題をひとまとめで一気に

In [ ]:
# ロジスティック回帰,SVM, 決定木, ランダムフォレスト, k最近傍法をひとまとめで実行してF1値を比較

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

# 各モデルのインスタンスをリストに格納
models = [
    LogisticRegression(max_iter=1000),
    LinearSVC(max_iter=1000),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    KNeighborsClassifier()
]

# 評価比較用の空リストの用意
model_name = []
f1_scores = []

# 各モデルをfor文で順に取り出して実行
for model in models:
    # model_name = model.__class__.__name__  # モデル名を取得
    model.fit(X_train, y_train)  # モデルの学習
    y_pred = model.predict(X_test)  # テストデータで予測
    f1 = f1_score(y_test, y_pred, average="micro")  # F1値の計算
    model_name.append(model_name)  # モデル名をリストに追加
    f1_scores.append(f1)  # F1値をリストに追加
    print(f"{model_name} F1 Score: {f1:.4f}")  # 各モデルのF1値を表示

# F1値の比較結果を表示
for name, score in zip(model_name, f1_scores):
    print(f"{name}: {score:.4f}")


In [ ]:
# ロジスティック回帰,SVM, 決定木, ランダムフォレスト, k最近傍法をひとまとめで実行してF1値を比較

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import lightgbm as lgb

# データトラベルを作成
X = df1.drop('target', axis=1)  # 特徴量
y = df1['target']  # 目的変数

# データを訓練用とテスト用に分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 各種分類器の定義
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machine": SVC(probability=True),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "K Nearest Neighbors": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "LightGBM": lgb.LGBMClassifier()
}

# 各種分類器の訓練と評価
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)  # モデルの訓練
    y_pred_proba = clf.predict_proba(X_test)[:, 1]  # テストデータでの予測確率
    auc = roc_auc_score(y_test, y_pred_proba)  # AUCの計算
    print(f"{name} AUC: {auc:.4f}")  # AUCの表示


# 検定
## 1標本のt検定

In [ ]:
# 1標本のt検定
from scipu import stats
diff1 = df1['value1'] - df1['value2']
pop_mean = 0  # 帰無仮説の母平均
alpha = 0.05  # 有意水準

x_bar = np.mean(diff1)  # 標本平均
std = np.std(diff1, ddof=1)
n = len(diff1)  # 標本サイズ
degree_freedom = n - 1  # 自由度
u = np.std(diff1, ddof=1)
se = u / np.sqrt(n)  # 標準誤差
t_statistic = (x_bar - pop_mean) / se  # t値
p_value = stats.t.cdf(-np.abs(t_statistic), df=degree_freedom)  # 両側検定のp値
confidence_interval = stats.t.interval(1 - alpha, df=degree_freedom, loc=x_bar, scale=se)  # 信頼区間
ci_lower, ci_upper = confidence_interval

print("1標本t検定の結果")
print("標本平均: {:.4f}".format(x_bar))
print("標本標準偏差: {:.4f}".format(std))
print("標本サイズ: {}".format(n))
print("自由度: {}".format(degree_freedom))
print("t値: {:.4f}".format(t_statistic))
print("p値: {:.4f}".format(p_value))
print("95%信頼区間: [{:.4f}, {:.4f}]".format(ci_lower, ci_upper))
if p_value < alpha:
    print("帰無仮説を棄却します。")
else:
    print("帰無仮説を棄却できません。")


## welchのt検定

In [ ]:
# welchのt検定
# 対応のない2群で、n数が異なるあるいは等分散を仮定できない場合に利用

# read data
x1 = []
x2 = []
x1 = df1[df1['group'] == 'A']['value1'].tolist()
x2 = df1[df1['group'] == 'B']['value1'].tolist()


from scipy import stats
import numpy as np

alpha = 0.05  # 有意水準
n1, n2 = len(x1), len(x2)
s1, s2 = np.var(x1, ddof=1), np.var(x2, ddof=1) # 不偏分散
s = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2) * (1/n1 + 1/n2)) # pooled standard deviation
degree_freedom = (s1/n1 + s2/n2) ** 2 / ((s1/n1) ** 2 / (n1 - 1) + (s2/n2) ** 2 / (n2 - 1))  # 自由度
t = stats.t.ppf(1 - alpha/2, degree_freedom)  # t-critical value for 95% confidence interval 

diff_mean = np.mean(x1) - np.mean(x2)
ci_lower = (np.mean(x1) - np.mean(x2)) - t * np.sqrt(1/len(x1) + 1 / len(x2)) * s
ci_upper = (np.mean(x1) - np.mean(x2)) + t * np.sqrt(1/len(x1) + 1 / len(x2)) * s
ind_t_test = stats.ttest_ind(x1, x2, equal_var=False)  # Welchのt検定

print("Welchのt検定の結果")
print("群1の平均: {:.4f}".format(np.mean(x1)))
print("群2の平均: {:.4f}".format(np.mean(x2)))
print("平均の差: {:.4f}".format(diff_mean))
print("t値: {:.4f}".format(ind_t_test.statistic))
print("p値: {:.4f}".format(ind_t_test.pvalue))
print("95%信頼区間: [{:.4f}, {:.4f}]".format(ci_lower, ci_upper))
if ind_t_test.pvalue < alpha:
    print("帰無仮説を棄却します。")
else:
    print("帰無仮説を棄却できません。")


# t分布を描く

In [ ]:
# t分布
X_a = np.linspace(-4, 4, 100)
Y_a = stats.t.pdf(X_a, df=10, loc=0, scale=1)  # 自由度10
plt